# 注意力机制

## 第一阶段：最简单的注意力（无学习权重）

我们先实现一个“呆板”的会议，词与词之间的相关性直接用向量的点积来算，没有任何需要学习的参数。目的是让你看清数据流：嵌入向量 → 点积算相关性 → 概率化 → 加权求和得到新向量。

### 1. 准备输入：一个已经嵌入好的句子

In [2]:
import torch

# 6 个词，每个词向量维度为 3 -> 形状为 (6, 3)
inputs = torch.tensor([
    [0.43, 0.15, 0.89],  # Your
    [0.55, 0.87, 0.66],  # journey
    [0.57, 0.85, 0.64],  # starts
    [0.22, 0.58, 0.33],  # with
    [0.77, 0.25, 0.10],  # one
    [0.05, 0.80, 0.55]   # step
])
print(f"输入形状: {inputs.shape}")  # (6, 3)

输入形状: torch.Size([6, 3])


### 2. 计算注意力得分（相关性）

如何知道两个词的相关程度？用点积。两个向量方向越一致，值越大。
以第二个词 "journey" 作为查询，计算它和所有词（包括自己）的相关性分数。

In [3]:
# 选第 2 个词作为查询 (索引从0开始，所以是 inputs[1])
query = inputs[1]
# 初始化一个装6个分数的空数组
attn_scores = torch.empty(inputs.shape[0])


print(attn_scores)
print(query)
for i, word_vector in enumerate(inputs):
    # 点积: 对应位置相乘再相加
    
    # print(word_vector)
    
    attn_scores[i] = torch.dot(word_vector, query)
    print(attn_scores[i])

print(f"原始相关性分数: {attn_scores}")

tensor([-8.7767e-11,  1.9086e-42,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00])
tensor([0.5500, 0.8700, 0.6600])
tensor(0.9544)
tensor(1.4950)
tensor(1.4754)
tensor(0.8434)
tensor(0.7070)
tensor(1.0865)
原始相关性分数: tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


数据流关键点： 一行代码 inputs @ inputs.T 能同时算出所有词之间的分数，得到一个 (6, 6) 的“相关性矩阵”。第 i 行第 j 列表示第 i 个词对第 j 个词的相关性。

In [4]:
# 矩阵乘法一次性完成所有点积
attn_scores_mat = inputs @ inputs.T
print(f"相关性矩阵 (6x6):\n{attn_scores_mat}")

相关性矩阵 (6x6):
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [5]:
# 初始化一个装6个分数的空数组
attn_scores2 = torch.empty(inputs.shape[0])
attn_scores2 = inputs @ query  # (6, 3) @ (3,) → (6,)
print(f"原始相关性分数: {attn_scores2}")

原始相关性分数: tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


两个向量方向越一致，点积越大。如果它们指向完全相反的方向，点积会很小甚至为负。这样就能量化"journey"和"Your"在语义空间里的接近程度。
需要注意，torch.dot 只支持一维向量，多维矩阵运算得用 @ 或 torch.matmul。

### 3. 分数转概率：Softmax

原始分数数值范围不确定。我们需要把它们转化成概率（所有数在 0-1 之间，且总和为 1），让模型知道该放多大的“注意力”在各个词上。

In [6]:
# dim=-1 表示对每一行做 softmax
attn_weights = torch.softmax(attn_scores_mat, dim=-1)
print(f"注意力权重 (每行和为1):\n{attn_weights}")

注意力权重 (每行和为1):
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


### 4. 计算上下文向量

In [7]:
# 矩阵乘法: (6,6) @ (6,3) -> (6,3)
context_vectors = attn_weights @ inputs
print(f"带上下文的词向量:\n{context_vectors}")

带上下文的词向量:
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


至此的数据流管线：

## 第二阶段：可学习的自注意力（核心机制）

核心 PyTorch 实现

In [14]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):
    def __init__(self, embed_dim, d_out, bias=False):
        """
        embed_dim: 输入词向量的维度 (这里是3)
        d_out: 投影后的维度 (设为2，方便演示)
        """
        super().__init__()
        # 三个线性层，本质就是三个大矩阵 W_query, W_key, W_value
        self.W_query = nn.Linear(embed_dim, d_out, bias=bias)
        self.W_key   = nn.Linear(embed_dim, d_out, bias=bias)
        self.W_value = nn.Linear(embed_dim, d_out, bias=bias)

    def forward(self, x):
        # 1. 生成 Q, K, V: 6x3 的矩阵乘 3x2 的矩阵 -> 6x2

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # 2. 计算注意力分数: 6x2 @ 2x6 -> 6x6
        attn_scores = queries @ keys.T

        # 3. 缩放 + Softmax: 除以 sqrt(d_k) 防止数值过大
        d_k = keys.shape[-1]
        attn_weights = torch.softmax(attn_scores / (d_k ** 0.5), dim=-1)

        # 4. 加权求和: 6x6 @ 6x2 -> 6x2
        context_vec = attn_weights @ values
        return context_vec

# 初始化并测试
sa = SelfAttention_v2(embed_dim=3, d_out=2)
output = sa(inputs)
print(f"自注意力输出形状: {output.shape}")  # (6, 2)
print(f"输出向量:\n{output}")

自注意力输出形状: torch.Size([6, 2])
输出向量:
tensor([[ 0.0767, -0.1442],
        [ 0.0765, -0.1448],
        [ 0.0765, -0.1447],
        [ 0.0764, -0.1451],
        [ 0.0768, -0.1440],
        [ 0.0763, -0.1455]], grad_fn=<MmBackward0>)


forward 会自动执行。 这是 PyTorch nn.Module 的核心机制，不需要手动调用。

## 第三阶段：因果掩码（让目光只能看身后）

GPT 是逐个生成单词的，所以预测第 i 个词时，不能偷看第 i+1 个词。我们通过掩码把未来的注意力分数设成 -inf（负无穷大），这样进入 softmax 后这些位置的权重就变成 0。

数据流的关键一步就是在 softmax 之前用一个下三角矩阵把未来分数盖掉。

In [17]:
import torch
import torch.nn as nn

# ===== 1. 准备数据 =====
inputs = torch.tensor([
    [0.43, 0.15, 0.89],  # Your
    [0.55, 0.87, 0.66],  # journey
    [0.57, 0.85, 0.64],  # starts
    [0.22, 0.58, 0.33],  # with
    [0.77, 0.25, 0.10],  # one
    [0.05, 0.80, 0.55]   # step
])

# ===== 2. 复用之前定义的 SelfAttention_v2 =====
class SelfAttention_v2(nn.Module):
    def __init__(self, embed_dim, d_out, bias=False):
        super().__init__()
        self.W_query = nn.Linear(embed_dim, d_out, bias=bias)
        self.W_key   = nn.Linear(embed_dim, d_out, bias=bias)
        self.W_value = nn.Linear(embed_dim, d_out, bias=bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        attn_scores = queries @ keys.T
        d_k = keys.shape[-1]
        attn_weights = torch.softmax(attn_scores / (d_k ** 0.5), dim=-1)

        context_vec = attn_weights @ values
        return context_vec

sa = SelfAttention_v2(embed_dim=3, d_out=2)

# ===== 3. 先计算出 attn_scores 和 keys =====
# 手动执行 forward 中的前半部分
x = inputs
queries = sa.W_query(x)
keys    = sa.W_key(x)
values  = sa.W_value(x)

attn_scores = queries @ keys.T
print("注意力分数:\n", attn_scores)

# ===== 4. 加因果掩码 =====
context_length = attn_scores.shape[0]
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
print("\n掩码矩阵:\n", mask)

masked_scores = attn_scores.masked_fill(mask.bool(), -float('inf'))
print("\n掩码后的分数:\n", masked_scores)

# 现在 keys 已经定义好了
d_k = keys.shape[-1]
attn_weights_causal = torch.softmax(masked_scores / (d_k ** 0.5), dim=-1)
print("\n因果注意力权重 (每行只看当前及之前的位置):\n", attn_weights_causal)

注意力分数:
 tensor([[-0.1458, -0.2027, -0.2022, -0.1053, -0.1377, -0.1171],
        [-0.1194, -0.3094, -0.3015, -0.1981, -0.0738, -0.2847],
        [-0.1140, -0.3026, -0.2947, -0.1947, -0.0685, -0.2812],
        [-0.0671, -0.1816, -0.1768, -0.1173, -0.0394, -0.1701],
        [ 0.0150, -0.0945, -0.0885, -0.0791,  0.0455, -0.1397],
        [-0.1237, -0.2584, -0.2535, -0.1568, -0.0933, -0.2132]],
       grad_fn=<MmBackward0>)

掩码矩阵:
 tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])

掩码后的分数:
 tensor([[-0.1458,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.1194, -0.3094,    -inf,    -inf,    -inf,    -inf],
        [-0.1140, -0.3026, -0.2947,    -inf,    -inf,    -inf],
        [-0.0671, -0.1816, -0.1768, -0.1173,    -inf,    -inf],
        [ 0.0150, -0.0945, -0.0885, -0.0791,  0.0455,    -inf],
        [-0.1237, -0.2584, -0.2535

## 第四阶段：多头注意力（并行开多个不同侧重点的会议）

In [18]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # 每个头的维度

        self.W_query = nn.Linear(d_in, d_out)
        self.W_key = nn.Linear(d_in, d_out)
        self.W_value = nn.Linear(d_in, d_out)
        self.out_proj = nn.Linear(d_out, d_out)  # 最后的融合层
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, n_tokens, _ = x.shape  # b: 批大小, n: 词数

        # 1. 统一投影，然后切分多头 (数据变形)
        # 形状: (b, n, d_out) -> (b, n, 头数, 头维度)
        queries = self.W_query(x).view(b, n_tokens, self.num_heads, self.head_dim)
        keys    = self.W_key(x).view(b, n_tokens, self.num_heads, self.head_dim)
        values  = self.W_value(x).view(b, n_tokens, self.num_heads, self.head_dim)

        # 2. 交换维度方便矩阵运算: (b, 头数, n, 头维度)
        queries = queries.transpose(1, 2)
        keys    = keys.transpose(1, 2)
        values  = values.transpose(1, 2)

        # 3. 正常算注意力，和单头逻辑一样，只是现在有多组矩阵
        attn_scores = queries @ keys.transpose(2, 3)
        # 应用因果掩码
        mask_bool = self.mask.bool()[:n_tokens, :n_tokens]
        attn_scores.masked_fill_(mask_bool, -float('inf'))

        attn_weights = torch.softmax(attn_scores / (keys.shape[-1] ** 0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 4. 加权求和并合并多头
        context_vecs = (attn_weights @ values)  # (b, 头数, n, 头维度)
        context_vecs = context_vecs.transpose(1, 2).contiguous().view(b, n_tokens, -1)

        # 5. 最终线性映射，让不同头的信息交互融合
        context_vecs = self.out_proj(context_vecs)
        return context_vecs